# Homework 2.1 — Setting up a Data Lake in AWS

**Course:** AAI-540 · **Student:** *<your name here>* · **Date:** *<date>*

This notebook is the Homework 2.1 solution. It follows the same workflow as the guided lab
(**Lab 2.1 — Setting up a Data Lake in AWS**) and completes the graded exercise:

1. **Set Up Data Lake** — ingest the homework dataset (Spotify tracks, `data/dataset.csv`) into the S3 data lake.
2. **Set up the Athena query engine** — database, external table, and a Parquet conversion.
3. **Query the Data Lake** — run the 5 required queries with **SQL (Athena)** and **Pandas**, verifying both give the same answer.

**Query engines used:** Amazon Athena (SQL over S3) and AWS Data Wrangler (Pandas integration).

**Prerequisites (SageMaker Studio):**
- Run this notebook with the **Data Science** kernel (select it in the top-right kernel picker and wait for it to start).
- Both repos must be cloned: `aai-540-labs` (lab tutorials) and `aai-540-homework` (this notebook + dataset).
- Open this notebook from inside `aai-540-homework/homework-2-1/` so the relative path `data/dataset.csv` resolves.

**Dataset:** `data/dataset.csv` — 114,000 Spotify tracks with audio features (energy, danceability, popularity, etc.)
and a `track_genre` label. Columns: `row_index` (unnamed index column), `track_id`, `artists`, `album_name`,
`track_name`, `popularity`, `duration_ms`, `explicit`, `danceability`, `energy`, `key`, `loudness`, `mode`,
`speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, `time_signature`, `track_genre`.

> **Data lake layout produced by this notebook**
> - `s3://<default-bucket>/spotify/tracks/raw/` — the raw CSV as delivered (immutable bronze zone)
> - `s3://<default-bucket>/spotify/tracks/parquet/` — curated Parquet, partitioned by `track_genre` (silver zone)
> - Athena database `spotify_db` with tables `tracks_csv` (raw) and `tracks_parquet`, plus the typed view `tracks`


## 1. Setup

Install/upgrade the AWS tooling and the query libraries (AWS Data Wrangler + PyAthena), following Lab 2.1.
Ignore pip warnings — this is expected in SageMaker Studio, exactly as in the lab.


> **First run? Restart once.** The cell below upgrades AWS packages in-place. After it finishes, do **Kernel → Restart Kernel** and then continue from the imports cell — this guarantees the upgraded `boto3`/`botocore` and the `sagemaker` SDK load cleanly instead of half-initialized (a classic SageMaker Studio gotcha that otherwise shows up as `AttributeError: module 'sagemaker' has no attribute 'Session'`). On later full runs, the installs are no-ops and no restart is needed.


In [ ]:
%pip install --upgrade boto3 botocore awscli
%pip install --disable-pip-version-check -q awswrangler pyathena


In [ ]:
import boto3
import sagemaker
import pandas as pd

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")

s3 = boto3.Session().client(service_name="s3", region_name=region)

print("Default bucket: {}".format(bucket))


In [ ]:
# Health check: make sure the REAL SageMaker SDK is loaded.
# If this fails, a local file/folder may be shadowing 'sagemaker', or the kernel
# needs a restart after the pip upgrades above (Kernel -> Restart Kernel).
import importlib
importlib.reload(sagemaker)
assert hasattr(sagemaker, "Session"), (
    "'sagemaker' module is not the real SDK (loaded from: {}). "
    "Restart the kernel (Kernel -> Restart Kernel). If it persists, look for a file "
    "or folder literally named 'sagemaker' in this notebook's directory and rename it."
    .format(getattr(sagemaker, "__file__", "<unknown>"))
)
print("[OK] sagemaker SDK {} loaded from {}".format(sagemaker.__version__, sagemaker.__file__))


In [ ]:
# Verify the SageMaker default bucket exists (same check as Lab 2.1 "Create S3 Bucket")
from botocore.client import ClientError

response = None

try:
    response = s3.head_bucket(Bucket=bucket)
    print(response)
    print("[OK] Bucket {} is ready.".format(bucket))
except ClientError as e:
    print("[ERROR] Cannot find bucket {} in {} due to {}.".format(bucket, response, e))


## 2. Ingest the Homework Data into the S3 Data Lake

We upload the homework dataset from the cloned repo into the **raw zone** of the data lake.
If the local file is missing, we fall back to downloading it from the AAI-540 homework GitHub repository.


In [ ]:
import os

# Dataset ships with the homework repo next to this notebook
local_dataset_path = "data/dataset.csv"

if not os.path.exists(local_dataset_path):
    print("Local dataset not found - downloading from the AAI-540 homework GitHub repo...")
    !mkdir -p data
    !curl -sL -o data/dataset.csv https://raw.githubusercontent.com/mechristenson/aai-540-homework/main/homework-2-1/data/dataset.csv

print("Local dataset ready: {}".format(os.path.abspath(local_dataset_path)))


In [ ]:
s3_private_path_csv = "s3://{}/spotify/tracks/raw".format(bucket)
print(s3_private_path_csv)


In [ ]:
# Copy the CSV into the raw zone of the data lake (single file, ~11 MB - takes seconds)
!aws s3 cp $local_dataset_path $s3_private_path_csv/dataset.csv

# Verify
!aws s3 ls $s3_private_path_csv/


## 3. Set Up the Athena Query Engine — Create the Database

Same pattern as Lab 2.1 (`02_Create_Athena_Database`): PyAthena connection with an S3 staging
directory where Athena writes query results.


In [ ]:
from pyathena import connect

database_name = "spotify_db"

# S3 staging directory - temporary location used by Athena for query results
s3_staging_dir = "s3://{0}/athena/staging".format(bucket)

conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)


In [ ]:
statement = "CREATE DATABASE IF NOT EXISTS {}".format(database_name)
print(statement)
pd.read_sql(statement, conn)


In [ ]:
# Verify the database exists
statement = "SHOW DATABASES"

df_show = pd.read_sql(statement, conn)
df_show.head(5)


## 4. Register the Raw CSV Dataset with Athena

We register the raw CSV as an external table so Athena can query it in place, without moving data.

**Important data-engineering decision — CSV quoting.** Unlike the lab's TSV files, this CSV uses standard
RFC-4180 quoting, and **9,023 rows contain quoted commas or doubled quotes** inside fields, e.g.:

- `"Cover Sessions, Vol. 4"` (comma inside the album name)
- `"Speak Your Mind (From the Netflix Series ""We The People"")"` (doubled quotes inside the track name)

The lab's `LazySimpleSerDe` treats **every** comma as a column separator and does not understand quoting,
so it would shift the columns and corrupt those rows (wrong `energy`/`popularity` values, `NULL`s).
`OpenCSVSerde` parses CSV quoting correctly. Its trade-off is that **all columns come in as STRING**,
so we create a **typed view** on top of it in the next section and do all querying against the view
(or the Parquet table we build afterwards).


In [ ]:
table_name_csv = "tracks_csv"

statement = """CREATE EXTERNAL TABLE IF NOT EXISTS {}.{}(
         row_index string,
         track_id string,
         artists string,
         album_name string,
         track_name string,
         popularity string,
         duration_ms string,
         explicit string,
         danceability string,
         energy string,
         key string,
         loudness string,
         mode string,
         speechiness string,
         acousticness string,
         instrumentalness string,
         liveness string,
         valence string,
         tempo string,
         time_signature string,
         track_genre string
) ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
   'separatorChar' = ',',
   'quoteChar'     = '\"',
   'escapeChar'    = '\"'
) LOCATION '{}'
TBLPROPERTIES ('skip.header.line.count'='1')""".format(
    database_name, table_name_csv, s3_private_path_csv
)

print(statement)
pd.read_sql(statement, conn)


In [ ]:
statement = "SHOW TABLES in {}".format(database_name)

df_show = pd.read_sql(statement, conn)
df_show.head(5)


In [ ]:
# Peek at the raw table - everything is still a string
statement = "SELECT * FROM {}.{} LIMIT 5".format(database_name, table_name_csv)

df = pd.read_sql(statement, conn)
df.head(5)


## 5. Create a Typed View over the Raw Table

The view casts every column to its proper type. The `WHERE TRY_CAST(popularity AS INT) IS NOT NULL`
guard makes the view robust: even if header skipping behaved unexpectedly, the header row (whose
`popularity` value is the literal string `'popularity'`) can never leak into query results.


In [ ]:
statement = """CREATE OR REPLACE VIEW {}.tracks AS
SELECT
         CAST(row_index AS INT) AS row_index,
         track_id,
         artists,
         album_name,
         track_name,
         CAST(popularity AS INT) AS popularity,
         CAST(duration_ms AS BIGINT) AS duration_ms,
         explicit,
         CAST(danceability AS DOUBLE) AS danceability,
         CAST(energy AS DOUBLE) AS energy,
         CAST(key AS INT) AS key,
         CAST(loudness AS DOUBLE) AS loudness,
         CAST(mode AS INT) AS mode,
         CAST(speechiness AS DOUBLE) AS speechiness,
         CAST(acousticness AS DOUBLE) AS acousticness,
         CAST(instrumentalness AS DOUBLE) AS instrumentalness,
         CAST(liveness AS DOUBLE) AS liveness,
         CAST(valence AS DOUBLE) AS valence,
         CAST(tempo AS DOUBLE) AS tempo,
         CAST(time_signature AS INT) AS time_signature,
         track_genre
FROM {}.{}
WHERE TRY_CAST(popularity AS INT) IS NOT NULL""".format(database_name, database_name, table_name_csv)

print(statement)
pd.read_sql(statement, conn)


In [ ]:
# Sanity check: the view must contain exactly the 114,000 data rows (no header, no corruption)
statement = "SELECT COUNT(*) AS n_rows FROM {}.tracks".format(database_name)

df = pd.read_sql(statement, conn)
print(df)

assert int(df["n_rows"][0]) == 114000, "Row count mismatch - check the raw table parsing!"
print("[OK] All 114,000 rows parsed correctly.")


## 6. Convert to Parquet with Athena (CTAS) — the Curated Zone

Following Lab 2.1 (`04_Convert_S3_TSV_To_Parquet_With_Athena`), we use an Athena **CTAS** query to convert
the data to columnar Parquet, **partitioned by `track_genre`** (114 genres). Parquet is columnar and
compressed, so scans are faster and cheaper than reading the raw CSV.

The `DROP TABLE IF EXISTS` first makes the cell safe to re-run (for a CTAS table, DROP TABLE also
deletes the underlying data in S3).


In [ ]:
table_name_parquet = "tracks_parquet"
s3_path_parquet = "s3://{}/spotify/tracks/parquet".format(bucket)

# Safe to re-run: drop the previous CTAS table (and its data) if present
statement = "DROP TABLE IF EXISTS {}.{}".format(database_name, table_name_parquet)
pd.read_sql(statement, conn)

statement = """CREATE TABLE {}.{}
WITH (format = 'PARQUET', external_location = '{}', partitioned_by = ARRAY['track_genre']) AS
SELECT row_index,
         track_id,
         artists,
         album_name,
         track_name,
         popularity,
         duration_ms,
         explicit,
         danceability,
         energy,
         key,
         loudness,
         mode,
         speechiness,
         acousticness,
         instrumentalness,
         liveness,
         valence,
         tempo,
         time_signature,
         track_genre
FROM {}.tracks""".format(
    database_name, table_name_parquet, s3_path_parquet, database_name
)

print(statement)
pd.read_sql(statement, conn)


In [ ]:
# Load the partition metadata (required after CTAS, same gotcha as the lab)
statement = "MSCK REPAIR TABLE {}.{}".format(database_name, table_name_parquet)
print(statement)
pd.read_sql(statement, conn)


In [ ]:
statement = "SHOW PARTITIONS {}.{}".format(database_name, table_name_parquet)

df_partitions = pd.read_sql(statement, conn)
print("Number of partitions: {}".format(len(df_partitions)))
df_partitions.head(10)


In [ ]:
%%time
# Parquet query with partition pruning - only the 'pop' partition is scanned
genre = "pop"

statement = """SELECT track_name, popularity FROM {}.{}
    WHERE track_genre = '{}' ORDER BY popularity DESC LIMIT 10""".format(
    database_name, table_name_parquet, genre
)

df = pd.read_sql(statement, conn)
df


## 7. Query the Data Lake with AWS Data Wrangler

As in Lab 2.1 (`05_Query_Data_With_AWS_DataWrangler`), Data Wrangler (`awswrangler`) is the second
query engine: it reads the Parquet dataset straight from S3 with partition push-down, and can also
run Athena SQL.


In [ ]:
import awswrangler as wr

path_parquet = "s3://{}/spotify/tracks/parquet/".format(bucket)

# Partition push-down: read only the 'pop' partition
p_filter = lambda x: x["track_genre"] == "pop"

df_pop = wr.s3.read_parquet(
    path_parquet,
    columns=["track_name", "popularity", "track_genre"],
    partition_filter=p_filter,
    dataset=True,
)
df_pop.shape


In [ ]:
df_pop.sort_values("popularity", ascending=False).head(5)


In [ ]:
# List the tables in the Glue catalog for our database
for table in wr.catalog.get_tables(database=database_name):
    print(table["Name"])


In [ ]:
%%time
df_wr = wr.athena.read_sql_query(
    sql="SELECT * FROM {} WHERE track_genre = 'pop' LIMIT 5000".format(table_name_parquet),
    database=database_name,
)
df_wr.head(5)


## 8. Homework Queries — SQL (Athena) and Pandas

Each question is answered **twice** and the two answers are verified to match:

1. **SQL** — executed on the Athena `tracks_parquet` table via `pd.read_sql(statement, conn)` (PyAthena), exactly in the style of the assignment's example snippets.
2. **Pandas** — executed on `df_tracks`, a DataFrame loaded once from the curated Parquet dataset in S3 via AWS Data Wrangler (this is the Pandas-side "query engine" on the data lake).

**Warm-up example from the assignment** — *show `track_name` and `energy` of tracks with energy above 0.5*:


In [ ]:
# SQL answer (warm-up)
statement = """
SELECT
track_name,
energy
FROM {}.{}
WHERE energy >= 0.5
""".format(database_name, table_name_parquet)

df_warmup_sql = pd.read_sql(statement, conn)
df_warmup_sql


In [ ]:
# Pandas answer (warm-up)
# Load the full dataset once for the Pandas answers - 114k rows is small enough
# to hold in memory (the lab warned about full loads for multi-GB tables; this is ~11 MB).
df_tracks = wr.s3.read_parquet(path_parquet, dataset=True)
print(df_tracks.shape)

df_warmup_pd = df_tracks[df_tracks["energy"] >= 0.5]
df_warmup_pd = df_warmup_pd[["track_name", "energy"]]
df_warmup_pd


---
### Query 1 — List `artist`, `track_name`, and `popularity` for songs with a popularity greater than or equal to 99


In [ ]:
# SQL answer
statement = """
SELECT
artists,
track_name,
popularity
FROM {}.{}
WHERE popularity >= 99
""".format(database_name, table_name_parquet)

df_q1_sql = pd.read_sql(statement, conn)
df_q1_sql


In [ ]:
# Pandas answer
df_q1_pd = df_tracks[df_tracks["popularity"] >= 99]
df_q1_pd = df_q1_pd[["artists", "track_name", "popularity"]]
df_q1_pd


In [ ]:
# Verify SQL and Pandas agree
pd.testing.assert_frame_equal(
    df_q1_sql.sort_values(["popularity", "track_name"]).reset_index(drop=True),
    df_q1_pd.sort_values(["popularity", "track_name"]).reset_index(drop=True),
)
print("[MATCH] SQL and Pandas returned the same 3 tracks.")


**Analysis.** Only **3 rows** in the whole dataset reach popularity ≥ 99, and the maximum value is 100
(Spotify's own popularity scale). The hits are *Unholy (feat. Kim Petras)* by Sam Smith;Kim Petras (100, listed
twice — once under the `dance` and once under the `pop` genre tag) and *Quevedo: Bzrp Music Sessions, Vol. 52*
by Bizarrap;Quevedo (99). This matches the dataset's late-2022 snapshot, when *Unholy* and the Bizarrap session
were topping the Spotify charts. Note the `artists` column uses a `;` separator, and a song appears once per
`track_genre` label, which is why the same track can show up more than once.


---
### Query 2 — List artists with an average popularity of 92


In [ ]:
# SQL answer
statement = """
SELECT
artists,
AVG(popularity) AS avg_popularity
FROM {}.{}
GROUP BY artists
HAVING AVG(popularity) = 92
""".format(database_name, table_name_parquet)

df_q2_sql = pd.read_sql(statement, conn)
df_q2_sql


In [ ]:
# Pandas answer
avg_popularity = df_tracks.groupby("artists")["popularity"].mean()

df_q2_pd = avg_popularity[avg_popularity == 92].rename("avg_popularity").reset_index()
df_q2_pd


In [ ]:
# Verify SQL and Pandas agree (round to 6 decimals to compare doubles safely)
pd.testing.assert_frame_equal(
    df_q2_sql.assign(avg_popularity=df_q2_sql["avg_popularity"].round(6)).sort_values("artists").reset_index(drop=True),
    df_q2_pd.assign(avg_popularity=df_q2_pd["avg_popularity"].round(6)).sort_values("artists").reset_index(drop=True),
)
print("[MATCH] Both engines found the same 2 artists.")


**Analysis.** Exactly **2 artists** average a popularity of 92: **Harry Styles** (3 tracks summing to 276)
and **Rema;Selena Gomez** (a single track, *Calm Down*, at 92 — a one-track "average").
An exact-equality threshold like `HAVING AVG(popularity) = 92` is brittle in real pipelines because averages are
floating-point numbers; a more robust filter would be `HAVING AVG(popularity) BETWEEN 91.5 AND 92.5` (or
`ROUND(AVG(popularity)) = 92`). It works here because the averages land exactly on 92.


---
### Query 3 — List the Top 10 genres with the highest average energy


In [ ]:
# SQL answer
statement = """
SELECT
track_genre,
AVG(energy) AS avg_energy
FROM {}.{}
GROUP BY track_genre
ORDER BY avg_energy DESC
LIMIT 10
""".format(database_name, table_name_parquet)

df_q3_sql = pd.read_sql(statement, conn)
df_q3_sql


In [ ]:
# Pandas answer
df_q3_pd = (
    df_tracks.groupby("track_genre")["energy"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .rename("avg_energy")
    .reset_index()
)
df_q3_pd


In [ ]:
# Verify SQL and Pandas agree
pd.testing.assert_frame_equal(
    df_q3_sql.assign(avg_energy=df_q3_sql["avg_energy"].round(6)).reset_index(drop=True),
    df_q3_pd.assign(avg_energy=df_q3_pd["avg_energy"].round(6)).reset_index(drop=True),
)
print("[MATCH] Top-10 genre ranking is identical in SQL and Pandas.")


**Analysis.** The top 10 is dominated by **loud, aggressive genres**: metal subgenres
(death-metal 0.931, grindcore 0.924, metalcore 0.914, black-metal 0.875, heavy-metal 0.874) and
high-tempo electronic styles (hardstyle 0.901, drum-and-bass 0.877, plus the `happy`, `party` and `j-idol`
playlist genres). This is consistent with how Spotify computes **energy**: fast tempo, loudness and distorted
guitars/synths drive the score up, so metal and EDM naturally cluster at the top while acoustic/folk genres
sit at the bottom. The gap between #10 (j-idol, 0.8687) and #11 (industrial, 0.8617) means the `LIMIT 10`
cut-off is unambiguous here.


---
### Query 4 — How many tracks is Bad Bunny on?


In [ ]:
# SQL answer
statement = """
SELECT
COUNT(*) AS bad_bunny_track_count
FROM {}.{}
WHERE LOWER(artists) LIKE '%bad bunny%'
""".format(database_name, table_name_parquet)

df_q4_sql = pd.read_sql(statement, conn)
df_q4_sql


In [ ]:
# Pandas answer (case-insensitive substring match, same semantics as the SQL)
df_q4_pd = df_tracks["artists"].str.contains("Bad Bunny", case=False).sum()
print("Bad Bunny track count: {}".format(df_q4_pd))


In [ ]:
# Verify SQL and Pandas agree
assert int(df_q4_sql["bad_bunny_track_count"][0]) == int(df_q4_pd)
print("[MATCH] SQL and Pandas both count {} tracks.".format(df_q4_pd))


**Analysis.** **Bad Bunny appears on 416 tracks** in this dataset (the dataset is from his chart-dominant era,
when he was Spotify's most-streamed artist globally). We match with a case-insensitive substring search so solo
tracks (`Bad Bunny`) and collaborations (`Bad Bunny;Jhay Cortez`, `KAROL G;Bad Bunny;...`) all count — the
`LOWER(...) LIKE '%bad bunny%'` in SQL and `str.contains(..., case=False)` in Pandas have identical semantics.


---
### Query 5 — Show the top 10 genres in terms of popularity, sorted by their most popular track


In [ ]:
# SQL answer: most popular track per genre
statement = """
SELECT
track_genre,
MAX(popularity) AS max_popularity
FROM {}.{}
GROUP BY track_genre
ORDER BY max_popularity DESC
LIMIT 10
""".format(database_name, table_name_parquet)

df_q5_sql = pd.read_sql(statement, conn)
df_q5_sql


In [ ]:
# Pandas answer
df_q5_pd = (
    df_tracks.groupby("track_genre")["popularity"]
    .max()
    .sort_values(ascending=False)
    .head(10)
    .rename("max_popularity")
    .reset_index()
)
df_q5_pd


In [ ]:
# Verify SQL and Pandas agree (sort by value + name so ties compare deterministically)
pd.testing.assert_frame_equal(
    df_q5_sql.sort_values(["max_popularity", "track_genre"]).reset_index(drop=True),
    df_q5_pd.sort_values(["max_popularity", "track_genre"]).reset_index(drop=True),
)
print("[MATCH] SQL and Pandas agree on the top-10 genres.")


**Analysis.** "Sorted by their most popular track" translates to: per genre, take the **maximum popularity**
(`MAX(popularity)`), then rank genres by that value. The result shows **dance and pop (100)** — carried by
*Unholy* — ahead of **hip-hop (99)** and a five-way tie at **98** (edm, latin, latino, reggae, reggaeton),
followed by **piano and rock (96)**. Latin-flavoured genres are strongly represented because Bad Bunny's and
similar artists' chart peak falls in the snapshot window. One nuance: ties mean the exact 10th slot could vary
between engines if there were more than two genres at 96 — here #10 (rock/piano, 96) is safely above #11
(alt-rock/alternative/chill/garage, 93), so `LIMIT 10` is deterministic.


## 9. Summary and Submission Checklist

**Data lake built**

| Object | Location |
|---|---|
| Raw CSV (bronze) | `s3://<default-bucket>/spotify/tracks/raw/dataset.csv` |
| Parquet, partitioned by `track_genre` (silver) | `s3://<default-bucket>/spotify/tracks/parquet/` |
| Athena database / tables | `spotify_db.tracks_csv` (raw), `spotify_db.tracks` (typed view), `spotify_db.tracks_parquet` |
| Query engines | Amazon Athena (SQL) + AWS Data Wrangler (Pandas) |

**Query results**

| # | Question | Answer |
|---|---|---|
| 1 | Songs with popularity ≥ 99 | 3 rows (Unholy ×2 at 100, Quevedo: Bzrp Music Sessions at 99) |
| 2 | Artists with average popularity 92 | Harry Styles; Rema;Selena Gomez |
| 3 | Top 10 genres by average energy | death-metal, grindcore, metalcore, happy, hardstyle, drum-and-bass, black-metal, heavy-metal, party, j-idol |
| 4 | Bad Bunny track count | 416 |
| 5 | Top 10 genres by their most popular track | dance 100, pop 100, hip-hop 99, edm/latin/latino/reggae/reggaeton 98, piano/rock 96 |

**For the PDF deliverable, screenshot each of these (query code + output visible):**
1. Section 8 warm-up example — SQL answer and Pandas answer.
2. Query 1 — SQL cell + result, Pandas cell + result.
3. Query 2 — SQL cell + result, Pandas cell + result.
4. Query 3 — SQL cell + result, Pandas cell + result.
5. Query 4 — SQL cell + result, Pandas cell + result.
6. Query 5 — SQL cell + result, Pandas cell + result.

Every SQL/Pandas pair is followed by an automated `[MATCH]` verification cell — include those in the screenshots
as evidence that both query engines agree.


In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>

<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}
</script>
